### This script creates a OAAT ensemble of parameter files, reading from a csv file provided and using a pre-defined base file. 

##### In this script we:
1) read in the csv file and make it into a data frame (df)
2) Modify the base parameter file with whatever common modifications we want all of the ensemble members to possess
3) Create and modify minimum and maximum files for each parameter.
4) There is different logic for those which have +- 50 or 100% modification, those that have defined min and maxes and those that have PFT specific ranges, which are defined in their own pft specific parmeter file


In [1]:
import shutil
import os
import netCDF4 as nc
import pandas as pd

##### Control variables and paths 

In [18]:
# Base parameter file, from previous calibration exercise
baseline_param_files = '/glade/u/home/rfisher/rosiefork_fates_global_cal/rosie_ppe/gaussian_emulator/parameter_outputs/manual_nocomp_calibration__16july.nc'

# Where to find the list of parameters and their modifications
parameter_list_file = '../csvfiles_for_parameter_modification/NOCOMP_FIRE_OAAT_parameters_a.csv'

# Adjusted default x calibration file to make a reasonable baseline. 
modified_baseline_param_files='../intermediate_pfiles/modified_average_file_260924.nc'

# What is the path to the new parameter file ensemble 
pdirroot='../paramfiles/NOCOMP_OAAT/nocomp_oaat_fire_v1_'


#### Read the main parameter list CSV file into a pandas DataFrame

In [25]:
print(parameter_list_file)
df = pd.read_csv(parameter_list_file)
fates_parameter_list = df['fates_parameter'].tolist()
minp = df['min'].tolist() 
maxp = df['max'].tolist() 
dimp = df['dim'].tolist()

../csvfiles_for_parameter_modification/NOCOMP_FIRE_OAAT_parameters_a.csv


#### Read the PFT parameter list CSV file into a pandas DataFrame

In [22]:
print(pft_parameter_list)
dfp = pd.read_csv(pft_parameter_list)
pftindex = dfp['pft_index'].tolist()
pftparam = dfp['param'].tolist() 
pftmin = dfp['min'].tolist()
pftmax = dfp['max'].tolist()

NameError: name 'pft_parameter_list' is not defined

#### map from the variables in the pftvariable csv file into the fates pft space

In [ ]:
string_mapping = {
    "fates_rad_stem_rhovis": "rhosvis",
    "fates_rad_stem_rhonir": "rhosnir",
    "fates_rad_leaf_rhonir": "rholnir",
    "fates_rad_leaf_rhovis": "rholvis",
    "fates_rad_leaf_taunir": "taulnir",
    "fates_rad_leaf_tauvis": "taulvis",
    "fates_leaf_stomatal_slope_medlyn": "medlynslope",
    "fates_leaf_slatop": "sla",
    "fates_rad_leaf_xl": "xl"   
}

#### Modify average file for NOCOMP specific baseline modifications


In [9]:
shutil.copy(baseline_param_files, modified_baseline_param_files)
nc_file = nc.Dataset(modified_baseline_param_files, 'r+')  # Open the file in write mode


rm = nc_file.variables['fates_maintresp_leaf_model'][:]
rm_new = rm
rm_new = 2
nc_file.variables['fates_maintresp_leaf_model'][ :] = rm_new  # Assign the modified values to the variable

cushion = nc_file.variables['fates_alloc_storage_cushion'][:]
cushion_new = cushion
cushion_new = cushion_new*0+2.4
cushion_new[0]=1.2 
cushion_new[3]= 1.2       
nc_file.variables['fates_alloc_storage_cushion'][ :] = cushion_new  # Assign the modified values to the variable

#make FATES CE negative to speed up runs.
fates_comp_excln = nc_file.variables['fates_comp_excln'][:]
cenew = fates_comp_excln *0 -1
nc_file.variables['fates_comp_excln'][ :] = cenew # Assign the modified values to the variable

#make FATES Crad model twostream
fates_rad_model = nc_file.variables['fates_rad_model'][:]
rmnew = fates_rad_model *0 +2
nc_file.variables['fates_rad_model'][:] = rmnew # Assign the modified values to the variable

print(nc_file['fates_rad_model'][:])
print(rmnew)


nc_file.close()

2.0
2.0


In [10]:
nc_file = nc.Dataset(modified_baseline_param_files, 'r+')  

print(nc_file['fates_rad_model'][:])
nc_file.close()

2.0


#### Retreive the full list of the parameters 

In [11]:
reference_vars = nc.Dataset(modified_baseline_param_files, 'r+') 
fatespfts=reference_vars.dimensions['fates_pft']
print(fatespfts.size)

modified_baseline_param_files

12


'../intermediate_pfiles/modified_average_file_260924.nc'

#### Modify variables in OAAT ensemble according to the values in the input csv fil

In [26]:
counter = 0
import numpy as np
for index, parameter in enumerate(fates_parameter_list):
    if parameter in reference_vars.variables:
        # Perform operations on each parameter
        print('#',index, parameter)  # Replace with your operations
       # print('minp index',minp[index])
#-------------------------------------------------------
        # Make minimum values    
        counter = counter + 1
        newfile = pdirroot + str(counter) + '.nc'    
        shutil.copy(modified_baseline_param_files, newfile)
        nc_file = nc.Dataset(newfile, 'r+')
        variable_to_modify = nc_file.variables[parameter]
            
        if minp[index] == '20percent':
            if variable_to_modify.dtype == int:
                print('integer')
                variable_data = variable_to_modify[:]
                variable_data = (variable_data * 0.8).astype(int)
                variable_to_modify[:] = variable_data  
                #print(variable_data)
            else:
                variable_to_modify[:] *= 0.8    
                print('pft min',variable_to_modify[:])
        elif minp[index] == '100percent':
                variable_to_modify[:] *= 0            
        elif minp[index] ==  'pft':
            shortvar = string_mapping[parameter]
            table = dfp[dfp['param'] == shortvar]
            pftmins=table['min'].tolist()
            variable_to_modify[:] *=  0
           
            for p in range(1, 12 ):               
                variable_to_modify[p-1] += float(pftmins[p-1])
            #variable_to_modify[12] += float(pftmins[10])
            #variable_to_modify[13] += float(pftmins[11])
            print(shortvar, ' table ',pftmins)
            print(shortvar, counter ,'min ',variable_to_modify[:])
        else:
            print('minp',minp[index])
            variable_to_modify[:] *=  0
            variable_to_modify[:] += float(minp[index])
        
        nc_file.close()  # Close the modified file
        
#-------------------------------------------------------
        # Make maximum values    
        counter = counter + 1
        newfile = pdirroot + str(counter) + '.nc'    
        shutil.copy(modified_baseline_param_files, newfile)
        nc_file = nc.Dataset(newfile, 'r+')
        variable_to_modify = nc_file.variables[parameter]
        if minp[index] == '20percent':
            if variable_to_modify.dtype == int:
                print('integer')
                variable_data = variable_to_modify[:]
                variable_data = (variable_data * 1.2).astype(int)
                variable_to_modify[:] = variable_data 
                print(variable_data)
            else:
                variable_to_modify[:] *= 1.2 
                print('pftmax',variable_to_modify[:])
        elif minp[index] == '100percent':
                variable_to_modify[:] = 1 
                print('100%',counter,parameter)
            
        elif minp[index] ==  'pft':
            shortvar = string_mapping[parameter]
            table = dfp[dfp['param'] == shortvar]
            pftmaxs=table['max'].tolist()
            variable_to_modify[:] *=  0
           
            for p in range(1, 12 ):
                variable_to_modify[p-1] += float(pftmaxs[p-1])
            #variable_to_modify[12] += float(pftmaxs[10])
            #variable_to_modify[13] += float(pftmaxs[11])
            print(shortvar, ' table ',pftmaxs)
            print(shortvar, counter, 'max ',variable_to_modify[:])
        else:
            print('maxp',maxp[index])
            variable_to_modify[:] *=  0
            variable_to_modify[:] += float(maxp[index])


        nc_file.close()  # Close the modified file

    else: #doenst exist 
        print('variable is not in file', parameter)

# 0 fates_alloc_storage_cushion
pft min [0.96 1.92 1.92 0.96 1.92 1.92 1.92 1.92 1.92 1.92 1.92 1.92]
pftmax [1.44 2.88 2.88 1.44 2.88 2.88 2.88 2.88 2.88 2.88 2.88 2.88]
# 1 fates_allom_d2bl2
pft min [1.28157432 1.21874984 1.04409896 1.56971176 1.28157432 1.11991512
 1.04       1.04       1.04       1.04       1.04       1.04      ]
pftmax [1.92236148 1.82812476 1.56614844 2.35456764 1.92236148 1.67987268
 1.56       1.56       1.56       1.56       1.56       1.56      ]
# 2 fates_allom_d2ca_coefficient_max
pft min [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
pftmax [0.32590692 0.44324616 1.29447108 0.06951564 0.32590692 1.38643344
 0.78821568 0.78821568 0.78821568 0.78821568 0.78821568 0.78821568]
# 3 fates_allom_d2ca_coefficient_min
pft min [0.21727128 0.29549744 0.86298072 0.04634376 0.21727128 0.92428896
 0.52547712 0.52547712 0.52547712 0.52547712 0.52547712 0.52547712]
pftmax [0.32590692 0.44324616 1.29447108 0.06951564 0.32590692 1.38643344
 0.78821568 0.78821568 0.78821568 0.7882156

In [43]:
newfile = pdirroot + str(80) + '.nc' 
nc_file = nc.Dataset(newfile, 'r+')
#print(nc_file.variables['fates_mort_ip_size_senescence'])
print(nc_file.variables['fates_fire_threshold'][:])
nc_file.close() 

50.0
